Assignment For Vehicle Monitoring Project

In [2]:
import cv2
import mediapipe as mp
import numpy as np

# Hand tracking setup
mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    max_num_hands=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# Webcam
cap = cv2.VideoCapture(0)

# Drawing canvas
canvas = None
prev_x, prev_y = None, None

try:
    while cap.isOpened():
        ret, frame = cap.read()

        if not ret:
            break

        # Mirror camera
        frame = cv2.flip(frame, 1)

        height, width, _ = frame.shape

        # Create blank canvas once
        if canvas is None:
            canvas = np.zeros_like(frame)

        # Convert BGR to RGB
        rgb_frame = cv2.cvtColor(
            frame, cv2.COLOR_BGR2RGB
        )

        # Detect hand
        results = hands.process(rgb_frame)

        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:

                # Draw hand landmarks
                mp_draw.draw_landmarks(
                    frame,
                    hand_landmarks,
                    mp_hands.HAND_CONNECTIONS
                )

                # Index finger tip = landmark 8
                index_tip = hand_landmarks.landmark[8]

                # Convert normalized coordinates to pixels
                x = int(index_tip.x * width)
                y = int(index_tip.y * height)

                # Check if index finger is raised
                index_pip = hand_landmarks.landmark[6]
                middle_tip = hand_landmarks.landmark[12]
                middle_pip = hand_landmarks.landmark[10]

                index_up = index_tip.y < index_pip.y
                middle_down = middle_tip.y > middle_pip.y

                # Draw only when index is up and middle is down
                if index_up and middle_down:

                    cv2.circle(
                        frame, (x, y), 8, (0, 255, 0), -1
                    )

                    if prev_x is not None:
                        cv2.line(
                            canvas,
                            (prev_x, prev_y),
                            (x, y),
                            (0, 0, 255),
                            5
                        )

                    prev_x, prev_y = x, y

                else:
                    # Stop drawing
                    prev_x, prev_y = None, None

        else:
            # Hand lost: stop the current stroke
            prev_x, prev_y = None, None

        # Combine camera and drawing canvas
        output = cv2.add(frame, canvas)

        cv2.imshow("Air Drawing", output)

        key = cv2.waitKey(1) & 0xFF

        if key == ord('q'):
            break

        elif key == ord('c'):
            canvas = np.zeros_like(frame)
            prev_x, prev_y = None, None

finally:
    cap.release()
    hands.close()
    cv2.destroyAllWindows()
